# CWE classification: flat vs graph traversal vs one-shot
This notebook is the experiment workflow, not a wrapper around a hidden CLI. Run cells in order (Restart Kernel and Run All). Reusable graph, scoring, artifact, and metric implementations remain in the adjacent Python modules and are unit tested. The compatibility CLI executes these same code cells.

**Default: offline synthetic smoke comparison, not a measured improvement.** No API calls, downloads, or credentials are needed for the default run. Change `demo` to false and supply an independently labeled benchmark for a real experiment. The default runs lexical flat/greedy/beam only; Chroma and Jev are opt-in. Real one-shot outputs must be supplied externally.

In [ ]:
import hashlib
import json
import math
import platform
import sys
import tempfile
from dataclasses import asdict
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# CLI supplies its root; Jupyter locates the checkout from the kernel directory.
ROOT = Path(globals()['experiment_root']) if 'experiment_root' in globals() else next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'experiments.ipynb').is_file() and (path / 'classification.py').is_file()), None)
if ROOT is None:
    raise ValueError('Launch Jupyter from the checkout or set experiment_root to its absolute path')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmark import load_benchmark, metrics, paired_intervals
from classification import CWEGraph, ChromaBaseline, classify, load_scorer, normalize_label
from experiment import catalog_fingerprint, create_one_shot_helper, export_prompts, fingerprint, load_one_shot, run_one_shot, validate_one_shot_config, write_jsonl


## 1. Freeze configuration before evaluation
Use dev data to choose margin, threshold, beam width, and Jev question formulation; do not tune on test results. Keep dev and test records in the same source file so the loader can detect text overlap across splits. `prepare` saves sanitized examples; `export-prompts` exports the full one-shot label list; `run` evaluates.

For Jev, install the `jev` extra, set `TYPESAFE_API_KEY` in the launching environment (never in cells or outputs), and select `jev_scorer:create_scorer`. See the adapter and README for the verified TypeSafe interface and model-pinning requirements. Input texts and CWE descriptions will be sent to TypeSafe only when you explicitly select that scorer; use public or approved data. Independent label questions are not candidate-relative Choice probabilities, but still require development-set calibration. Compare stopping policies on dev: `specificity` (default) forces descent through broad nodes and permits margin-based early stopping at Base/Variant, `improvement` reproduces the old parent/child rule, and `leaf` always descends through threshold-passing children. A blocked non-stoppable route abstains; an actual graph leaf can terminate.

No prompts are truncated. Verify the exported full prompt fits your LLM token budget; `max_prompt_chars` is only a character guard. Unknown training overlap is not evidence of a clean training split.

In [ ]:
config = dict(globals().get('experiment_config', {
    'command': 'run',
    'demo': True,
    'catalog': str(ROOT / 'data' / 'all.csv'),
    'benchmark': '/tmp/cwe-benchmark.jsonl',
    'dataset_manifest': None,
    'dataset_output': '/tmp/cwe-dataset',
    'test_after': None,
    'holdout_ecosystems': [],
    'holdout_vendors': [],
    'holdout_sources': [],
    'sample_size': 200,
    'max_examples_per_split': 300,
    'max_eval_examples': 300,
    'max_live_examples': 50,
    'max_candidate_evaluations': 1000000,
    'balanced_per_label': 20,
    'rare_max_count': 10,
    'near_duplicate_threshold': 0.9,
    'review_sample_size': 30,
    'output': '/tmp/cwe-notebook-results',
    'view': '1000',
    'split': 'test',
    'training_ids': None,
    'scorer': 'lexical',
    'jev_base_url': None,
    'jev_model': None,
    'methods': ['flat', 'greedy', 'beam'],
    'one_shot_predictions': None,
    'one_shot_provider': None,
    'one_shot_endpoint': None,
    'one_shot_model': None,
    'one_shot_api_key_env': None,
    'one_shot_timeout': 120,
    'beam_width': 3,
    'top_k': 5,
    'threshold': 0.0,
    'margin': 0.0,
    'stop_policy': 'specificity',
    'dev_fraction': 0.2,
    'include_observed_examples': False,
    'seed': 0,
    'bootstrap_samples': 2000,
    'max_prompt_chars': None,
}))
config.setdefault('stop_policy', 'specificity')
config.setdefault('dev_fraction', 0.2)
config.setdefault('include_observed_examples', False)
config.setdefault('max_eval_examples', 300)
config.setdefault('max_live_examples', 50)
config.setdefault('max_candidate_evaluations', 1000000)
for budget in ['max_eval_examples', 'max_live_examples', 'max_candidate_evaluations']:
    if isinstance(config[budget], bool) or not isinstance(config[budget], int) or config[budget] < 1:
        raise ValueError('Evaluation budgets must be positive integers')
if config['stop_policy'] not in {'specificity', 'improvement', 'leaf'} or not 0 <= config['dev_fraction'] < 1:
    raise ValueError('Invalid stopping policy or NVD dev fraction')
if config['command'] not in {'prepare', 'export-prompts', 'run', 'build-benchmark'}:
    raise ValueError('Unknown experiment command')
if (config['beam_width'] < 1 or config['top_k'] < 1 or config['bootstrap_samples'] < 1
        or config['margin'] < 0 or not math.isfinite(config['margin'])
        or not math.isfinite(config['threshold'])):
    raise ValueError('Invalid traversal or bootstrap parameters')
if not config['methods'] or not set(config['methods']) <= {'flat', 'greedy', 'beam', 'chroma', 'one-shot'}:
    raise ValueError('Choose supported experiment methods')
if config.get('demo') and (config['scorer'] != 'lexical' or set(config['methods']) - {'flat', 'greedy', 'beam'}):
    raise ValueError('Demo is offline lexical only; disable demo for real backends')
if config.get('one_shot_provider'):
    if config.get('demo'):
        raise ValueError('Demo is offline; disable demo before configuring a live provider')
    if config.get('one_shot_predictions'):
        raise ValueError('Choose live one_shot_provider OR imported one_shot_predictions')
    validate_one_shot_config(config)
elif any(config.get(key) is not None for key in ['one_shot_endpoint', 'one_shot_model', 'one_shot_api_key_env']):
    raise ValueError('Set one_shot_provider explicitly before configuring a transport')
print(json.dumps(config, indent=2))


## 2. Inspect the catalog and exclusion audit
### Snapshot manifest and source coverage
**Cost controls:** preparation defaults to 200 uniformly sampled natural groups per split and at most 300 scored groups total, including diagnostics. max_eval_examples caps any input benchmark; non-lexical classifier comparisons also default to max_live_examples=50 for an initial paid/local-model pilot. Increase that limit deliberately after reviewing usage and confidence intervals, rather than treating a 50-example pilot as precise population accuracy. Sampling is seeded, preserves natural examples first, and records population/selected label counts. max_candidate_evaluations preflights a conservative full-catalog bound across classifier methods before models are created; it is a work limit, not a USD estimate. Unknown pricing/token lengths cannot guarantee monetary cost. Keep identical budgets/settings when exporting one-shot prompts and running comparisons.

NVD covers broadly labeled CVEs; CVE List V5 preserves CNA wording; GitHub-reviewed advisories emphasize dependency vulnerabilities. These overlap and must not count as independent examples. Vendor JSONL provides longer/source-held-out descriptions; completed adjudicated JSONL supplies actual human gold; KEV supplies a known-exploitation diagnostic membership only. No source guarantees perfectly precise CWE labels.

Create a local manifest of the form `{"sources": [{"kind": "ghsa", "path": "/tmp/advisory-snapshot/advisories/github-reviewed", "revision": "<downloaded commit>", "sha256": "<snapshot_sha256 result>"}]}`. Other kinds are nvd, cve, vendor, adjudicated, kev. Each source is a file or extracted JSON-record directory. Call `dataset_sources.snapshot_sha256(Path(...))` to pin it; then preserve the manifest and snapshots. A checksum pins bytes but does not authenticate the publisher: obtain snapshots from the official sources and retain their download/commit provenance. Unreviewed/withdrawn/rejected records are not eligible primary evidence.

**Multi-source benchmark:** set dataset_manifest to a pinned local manifest and demo=False. The builder imports NVD, CVE List V5, GitHub-reviewed advisories, vendor/adjudicated JSONL, and KEV membership. It groups aliases transitively and conservative token-Jaccard near-duplicates before assigning splits or sampling. It excludes catalog/training overlap across all aliases. Conflicting source labels are quarantined unless completed expert adjudication resolves them; KEV membership never upgrades label confidence. No downloads, model calls, or human labels are fabricated.

Set command='build-benchmark' to export both splits, diagnostic groups, unresolved review_queue.jsonl, a blinded expert_review_sample.jsonl (up to review_sample_size=30 natural groups), and dataset-report.json to output. Reviewers must supply actual completed annotations; pending/suggested labels never become gold. Otherwise dataset_output holds these artifacts while run/export uses the selected split. Optional test_after, holdout_ecosystems/vendors/sources select a group-level generalization holdout (union); remaining groups are dev. Without holdouts, dev_fraction and seed assign groups. Missing publication dates are quarantined for temporal holdouts. Use sample_size for a uniform natural sample per split, balanced_per_label for a separate capped diagnostic subset, and rare_max_count for rarity based on dev labels only. Source/family breakdowns use the natural subset and may overlap; other diagnostic slice metrics are separate, not population estimates. Inspect the audit before interpreting representativeness: this is the eligible snapshot population, not all vulnerabilities.

The selected view filters ChildOf edges, not the whole catalog. Multiple parents and disconnected nodes are retained; obsolete/deprecated entries are excluded. JSONL needs explicit dev/test splits; an offline NVD dump uses a stable seeded CVE-ID hash for a dev/test split (default dev_fraction=0.2; zero retains test-only import). Sanitization removes CWE identifiers and deduplication excludes cross-split overlaps before selection. Catalog Observed Example CVEs are excluded by default with an audit count; include_observed_examples=True opts out for explicitly contaminated diagnostics. Known training IDs are still excluded separately. Gold labels are acceptable alternatives; unsupported alternatives are retained when at least one gold label is eligible.

The three demo examples below are handcrafted pipeline checks, not a held-out vulnerability dataset. Save the displayed original input hash and audit separately when using `prepare` or `export-prompts`; only `run` writes these in summary.json. For publication, evaluate the original combined benchmark, not just a prefiltered subset.

In [ ]:
graph = CWEGraph(config['catalog'], config['view'])
training_ids = None
if config.get('training_ids'):
    training_ids = json.loads(Path(config['training_ids']).read_text(encoding='utf-8'))
    if not isinstance(training_ids, list) or not all(isinstance(x, str) for x in training_ids):
        raise ValueError('training_ids must contain a JSON array of strings')
demo_directory = None
dataset_report = None
diagnostic_groups = []
review_queue = []
if config.get('dataset_manifest'):
    if config.get('demo') or config['include_observed_examples']:
        raise ValueError('Multi-source evaluation requires demo=False and catalog exclusions enabled')
    from datasets import build_benchmark
    dataset_options = {key: config[key] for key in ['seed', 'dev_fraction', 'test_after', 'holdout_ecosystems', 'holdout_vendors', 'holdout_sources', 'sample_size', 'max_examples_per_split', 'balanced_per_label', 'rare_max_count', 'near_duplicate_threshold', 'review_sample_size'] if key in config}
    dataset_options['training_ids'] = training_ids or []
    destination = config['output'] if config['command'] == 'build-benchmark' else config.get('dataset_output', '/tmp/cwe-dataset')
    diagnostic_groups, dataset_report, review_queue = build_benchmark(config['dataset_manifest'], graph.labels, destination, **dataset_options)
    benchmark_path = Path(destination) / 'benchmark.jsonl'
    print(json.dumps(dataset_report, indent=2))
elif config['command'] == 'build-benchmark':
    raise ValueError('build-benchmark requires dataset_manifest')
else:
    benchmark_path = Path(config['benchmark'])
if config.get('demo'):
    demo_directory = tempfile.TemporaryDirectory(prefix='cwe-notebook-')
    benchmark_path = Path(demo_directory.name) / 'demo.jsonl'
    write_jsonl(benchmark_path, [
        {'id': 'synthetic-sql', 'text': 'User input is concatenated into a SQL query and executed by the database.', 'labels': ['89'], 'split': 'test'},
        {'id': 'synthetic-xss', 'text': 'Untrusted text is rendered as HTML and runs script in a browser.', 'labels': ['79'], 'split': 'test'},
        {'id': 'synthetic-bounds', 'text': 'A memory write exceeds the allocated buffer boundary.', 'labels': ['787', '119'], 'split': 'test'},
    ])
raw_benchmark_hash = hashlib.sha256(benchmark_path.read_bytes()).hexdigest()
if config['command'] == 'build-benchmark':
    examples, audit = [], dataset_report['audit']
else:
    examples, audit = load_benchmark(benchmark_path, list(graph.labels), config['split'], training_ids,
        catalog=graph.labels, exclude_observed_examples=not config['include_observed_examples'],
        dev_fraction=config['dev_fraction'], seed=config['seed'])
for example in examples:
    example['labels'] = [normalize_label(label) for label in example['labels']]
nonlexical_comparison = config['scorer'] != 'lexical' and bool(set(config['methods']) & {'flat', 'greedy', 'beam'})
live_one_shot = config['command'] == 'run' and 'one-shot' in config['methods'] and bool(config.get('one_shot_provider'))
eval_limit = min(config['max_eval_examples'], config['max_live_examples']) if nonlexical_comparison or live_one_shot else config['max_eval_examples']
population_labels = dict(Counter(label for example in examples for label in example['labels']))
population_count = len(examples)
ranked = sorted(examples, key=lambda example: fingerprint([config['seed'], 'eval-budget', example['id']]))
if any('slices' in example for example in examples):
    ranked = [example for example in ranked if 'natural' in example['slices']] + [example for example in ranked if 'natural' not in example['slices']]
examples = sorted(ranked[:eval_limit], key=lambda example: example['id'])
evaluation_sampling = {'population_count': population_count, 'selected_count': len(examples), 'limit': eval_limit, 'population_label_counts': population_labels, 'selected_label_counts': dict(Counter(label for example in examples for label in example['labels'])), 'method': 'seeded uniform natural-first cap'}
print('Evaluation sampling:', json.dumps(evaluation_sampling, indent=2))
if demo_directory is not None:
    demo_directory.cleanup()
if not examples and config['command'] != 'build-benchmark':
    raise ValueError('No eligible examples remain; inspect benchmark and split')
print('Catalog labels:', len(graph.labels), 'Roots:', len(graph.roots), 'Examples:', len(examples))
print(json.dumps(audit, indent=2))
print('Original input SHA-256:', raw_benchmark_hash)
print(json.dumps(examples[:3], indent=2))


## 3. Optional benchmark preparation and one-shot export
Set `command` to `export-prompts` and `output` to a JSONL file. Existing genuine single-inference imports remain supported: preserve id/text_hash/catalog_hash, model, calls=1, truncated_labels=[], predictions (0 or 1 CWE), latency_seconds, and optional input_tokens/output_tokens/cost. Switch back to `run`, include `one-shot` in methods, and point `one_shot_predictions` to the output JSONL.

### Explicit live one-shot provider (approved/public inputs only)
The default demo remains offline. For live inference set `demo=False`, `command='run'`, and include `'one-shot'` in `methods`. Choose **either** imported `one_shot_predictions` **or** `one_shot_provider`. For a compatible server set `one_shot_provider='openai'`, `one_shot_endpoint='https://your-server.example/v1/chat/completions'` (full URL, not base URL), `one_shot_model='your-model'`, `one_shot_api_key_env='OPENAI_API_KEY'` (optional for an unauthenticated local server), and `one_shot_timeout=120`. Put the key only in the launching environment, never in notebook/config/artifacts. HTTPS is required except for loopback HTTP; credentials, queries and fragments in URLs are rejected. Redirects, retries, streaming and tools are not enabled. One HTTP request does not prove one server-internal inference call.

Alternatively set `one_shot_provider='copilot'` and `one_shot_model='your-supported-model'`, leaving endpoint/API-key-env unset. Install/authenticate GitHub Copilot CLI separately. The adapter uses one fresh empty system temporary working directory (keep TMPDIR outside the checkout), `-p`, `--model`, silent response-only text output, an empty `--available-tools` list, `--no-custom-instructions`, `--disable-builtin-mcps`, and `--no-ask-user`. It never resumes, enables fleet/autopilot, or grants allow-all permissions. Copilot is an agent runtime, **not** an OpenAI HTTP server: one prompt invocation is not a verified single inference call; there is no documented max-model-calls option. The empty tools syntax is verified with published CLI 1.0.95 help (`--available-tools [<tools>...]`); older incompatible versions fail without retry. CLI usage/resolved model/cost remain unknown.

**CLI runtime caveat:** a fresh cwd and disabled model tools/custom instructions are not a sandbox or proof of a pristine runtime. User-level `COPILOT_HOME` (default `~/.copilot`) can contain hooks, installed plugins, extensions and provider settings; the adapter does not clear that directory because it also stores authentication. Review/disable personal hooks and plugins before an approved run, and use a clean externally provisioned CLI configuration when reproducible isolation is required. These controls do not promise to suppress every runtime hook. Prefer the compatible HTTP provider for a direct completion without CLI runtime customizations. See the official [configuration directory reference](https://docs.github.com/en/copilot/reference/copilot-cli-reference/cli-config-dir-reference).

Notebook execution imports `create_one_shot_helper(config)` and `run_one_shot(examples, graph, path, helper, max_prompt_chars=None, max_live_examples=50)` from `experiment`. The helper is a replaceable callable `prompt -> completion mapping` containing content, model/requested_model/resolved_model, provider, latency_seconds, nullable input_tokens/output_tokens/cost/calls/model_requests, and explicit http_requests/prompt_invocations. To remove CLI logic later, use the openai provider or replace only this callable; prompt construction, evaluation and provenance are unchanged. New transport records deliberately keep calls/model_requests null and count HTTP requests or CLI invocations separately. Live examples use the same seeded `max_live_examples` cap as other live backends. Full prompts/config are preflighted before inference. Outputs save `one-shot-predictions.jsonl` and `one-shot.jsonl` with input/catalog/prompt hashes and unknown usage/cost preserved.

CLI equivalents are `--one-shot-provider {openai,copilot}`, `--one-shot-endpoint URL`, `--one-shot-model MODEL`, `--one-shot-api-key-env NAME`, `--one-shot-timeout SECONDS`, and `--max-live-examples N`; combine with `run --methods one-shot` and normal catalog/benchmark/output options. Export/import remains available without any live provider.

In [ ]:
if config['command'] == 'prepare':
    write_jsonl(config['output'], examples)
    print('Saved sanitized benchmark:', config['output'])
elif config['command'] == 'export-prompts':
    export_prompts(examples, graph, config['output'], config['max_prompt_chars'])
    print('Saved complete-label one-shot prompts:', config['output'])


## 4. Matched comparison: same text and eligible labels
Flat scores all labels. Greedy keeps one route; beam retains multiple active routes and deduplicates shared DAG nodes. Threshold filters roots and children. The specificity default prevents broad-node early stopping; improvement preserves the original rule and leaf removes parent-relative stopping. Scores must remain comparable for ranking across beam branches and any margin comparisons. Beam search may reduce early routing mistakes; this is a hypothesis, not a result. Terminal abstraction/path-length diagnostics reveal broad early stops.

Chroma uses the same descriptions/embedding approach as the original search, but a fresh ephemeral index and matched eligible catalog. Index initialization time is excluded from query latency. Selecting Chroma can download an embedding model. Scorer calls count adapter invocations; Jev's actual request counts and model metadata are separately captured when available.

In [ ]:
results = {}
scorer_metadata = {}
if config['command'] == 'run':
    classifier_methods = set(config['methods']) & {'flat', 'greedy', 'beam'}
    candidate_upper_bound = len(examples) * len(graph.labels) * len(classifier_methods)
    print('Conservative scorer candidate-evaluation bound:', candidate_upper_bound)
    if candidate_upper_bound > config['max_candidate_evaluations']:
        raise ValueError('Candidate evaluation budget exceeded; reduce examples/methods or explicitly raise max_candidate_evaluations')
    scorer = None
    if classifier_methods:
        if config['scorer'] == 'jev_scorer:create_scorer' and any(config.get(key) is not None for key in ['jev_base_url', 'jev_model']):
            from jev_scorer import JevScorer
            scorer = JevScorer(graph.labels, base_url=config.get('jev_base_url'), model=config.get('jev_model'))
        else:
            scorer = load_scorer(config['scorer'], graph.labels)
    chroma = None
    try:
        chroma = ChromaBaseline(graph.labels) if 'chroma' in config['methods'] else None
        scorer_metadata = dict(getattr(scorer, 'metadata', {}))
        if 'one-shot' in config['methods']:
            if config.get('one_shot_provider'):
                one_shot_helper = create_one_shot_helper(config)
                Path(config['output']).mkdir(parents=True, exist_ok=True)
                results['one-shot'] = run_one_shot(examples, graph, Path(config['output']) / 'one-shot-predictions.jsonl', one_shot_helper, config['max_prompt_chars'], config['max_live_examples'])
            elif config.get('one_shot_predictions'):
                results['one-shot'] = load_one_shot(config['one_shot_predictions'], examples, graph)
            else:
                raise ValueError('Supply one_shot_provider or genuine imported one_shot_predictions')
        for method in dict.fromkeys(config['methods']):
            if method == 'one-shot':
                continue
            records = []
            for example in examples:
                requests_before = getattr(scorer, 'requests', None)
                if method == 'chroma':
                    prediction = chroma.predict(example['text'], config['top_k'])
                else:
                    width = {'flat': None, 'greedy': 1, 'beam': config['beam_width']}[method]
                    prediction = classify(example['text'], graph, scorer, width, config['top_k'], config['threshold'], config['margin'], config['stop_policy'])
                record = {**example, **asdict(prediction)}
                if method != 'chroma' and requests_before is not None:
                    record['model_requests'] = scorer.requests - requests_before
                records.append(record)
            results[method] = records
            print(method, 'completed:', len(records), 'examples')
    finally:
        close_scorer = getattr(scorer, 'close', None)
        if callable(close_scorer):
            close_scorer()
    scorer_metadata = dict(getattr(scorer, 'metadata', {}))


## 5. Metrics, paired intervals, and reproducibility artifacts
**Top-1 accuracy and its paired intervals are the headline comparison**, accepting any annotated gold label. Top-k recall is descriptive only: greedy emits at most one prediction, beam emits a variable number, and flat/Chroma can emit k. Equal requested k does not imply equal returned list length; examine prediction_count_histogram and mean_prediction_count instead of treating top-5 differences as a fair ranking comparison. Custom macro-F1 has been removed. Distance is minimum undirected top-1-to-gold graph distance, excluding abstentions and separately counting disconnected predictions. Usage totals are null when any record is unknown; known subtotals do not imply a zero-cost run.

Paired percentile-bootstrap 95% intervals resample matched examples, using the configured seed. They do not correct training contamination or correlated duplicate vulnerabilities. Preserve the input data, lockfile, model revision, and settings when sharing results. Pin Jev rather than a moving model alias. The notebook records installed package versions and hashes but cannot prove remote-model determinism.

In [ ]:
summary = None
if config['command'] == 'run':
    packages = {}
    for package in ['chromadb', 'typesafe-sdk', 'jupyterlab', 'nbformat', 'nbclient']:
        try:
            packages[package] = version(package)
        except PackageNotFoundError:
            packages[package] = None
    traversal_diagnostics = {}
    for method in ['greedy', 'beam']:
        if method in results:
            records = results[method]
            traversal_diagnostics[method] = {
                'top1_abstraction_histogram': dict(Counter(graph.labels[r['predictions'][0]].get('Weakness Abstraction', 'Unknown') for r in records if r['predictions'])),
                'top1_path_length_histogram': dict(Counter(len(r['paths'][0]) for r in records if r['paths'])),
            }
    has_slices = any('slices' in example for example in examples)
    primary_results = {method: [record for record in records if not has_slices or 'natural' in record.get('slices', [])] for method, records in results.items()}
    slice_metrics = {}
    if has_slices:
        from datasets import evaluation_slices
        slice_metrics = evaluation_slices(results, graph, examples)
    summary = {
        'configuration': config,
        'synthetic_demo': bool(config.get('demo')),
        'catalog_hash': catalog_fingerprint(graph),
        'benchmark_hash': fingerprint(examples),
        'raw_benchmark_hash': raw_benchmark_hash,
        'workflow_hash': hashlib.sha256((ROOT / 'experiments.ipynb').read_bytes()).hexdigest(),
        'audit': audit,
        'dataset_report': dataset_report,
        'evaluation_sampling': evaluation_sampling,
        'candidate_evaluation_upper_bound': candidate_upper_bound,
        'diagnostic_counts': dict(Counter(name for record in diagnostic_groups if record['split'] == config['split'] for name in record['slices'])),
        'slice_metrics': slice_metrics,
        'primary_population': 'natural-distribution sampled groups' if has_slices else 'all eligible examples',
        'training_overlap': 'known IDs excluded' if training_ids is not None else 'unknown',
        'environment': {'python': platform.python_version(), 'packages': packages, 'lock_hash': hashlib.sha256((ROOT / 'uv.lock').read_bytes()).hexdigest()},
        'scorer_metadata': scorer_metadata,
        'headline_metric': 'exact_match_accuracy',
        'traversal_diagnostics': traversal_diagnostics,
        'metrics': {method: metrics(records, graph) for method, records in primary_results.items()},
        'paired_accuracy_intervals': paired_intervals(primary_results, config['seed'], config['bootstrap_samples']) if len(primary_results) > 1 and next(iter(primary_results.values())) else {},
        'notes': ['Synthetic/lexical checks are not evidence about learned classification.', 'Tune on dev only; freeze settings before test.', 'Top-1 is headline; returned list lengths vary, so top-k comparisons are descriptive only.', 'Setup/indexing time is excluded; unknown usage remains null.'],
    }
    output = Path(config['output'])
    output.mkdir(parents=True, exist_ok=True)
    for method, records in results.items():
        write_jsonl(output / f'{method}.jsonl', records)
    (output / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
    print(json.dumps(summary['metrics'], indent=2))
    print(json.dumps(summary['paired_accuracy_intervals'], indent=2))
    print(json.dumps(slice_metrics, indent=2))
    print('Artifacts:', output)


## 6. Inspect early routing errors
Look for examples where flat predicts an acceptable CWE but greedy does not, and whether beam recovers it. Paths reveal which branch was selected. These diagnostics are descriptive, not an invitation to retune on the test set. Do not publish notebook outputs containing private input texts, credentials, or sensitive paths.

In [ ]:
routing_errors = []
if config['command'] == 'run' and {'flat', 'greedy'} <= results.keys():
    for index, flat in enumerate(results['flat']):
        greedy = results['greedy'][index]
        flat_correct = bool(flat['predictions']) and flat['predictions'][0] in flat['labels']
        greedy_correct = bool(greedy['predictions']) and greedy['predictions'][0] in greedy['labels']
        if flat_correct and not greedy_correct:
            beam = results.get('beam', [])[index] if 'beam' in results else None
            routing_errors.append({'id': flat['id'], 'gold': flat['labels'], 'flat': flat['predictions'], 'greedy_paths': greedy['paths'], 'beam_paths': beam['paths'] if beam else None})
    print(json.dumps(routing_errors[:10], indent=2))
